#Combining the common cols in all tables into one table

In [0]:
import pyspark.sql.functions as F

In [0]:
yellow_input_path = 's3://ims-dbr-stage/yellow_trip_records/'
yellow_output_path = 's3://ims-dbr-published/yellow_trip_records/'

green_input_path = 's3://ims-dbr-stage/green_trip_records/'
green_output_path = 's3://ims-dbr-published/green_trip_records/'

fhv_input_path = 's3://ims-dbr-stage/fhv_trip_records/'
fhv_output_path = 's3://ims-dbr-published/fhv_trip_records/'

fhvhv_input_path = 's3://ims-dbr-stage/fhvhv_trip_records/'
fhvhv_output_path = 's3://ims-dbr-published/fhvhv_trip_records/'

zone_input_path = 's3://ims-dbr-published/zone_lookup/'

In [0]:
zone_df = spark.read.parquet(zone_input_path)
display(zone_df)

###creating new dataframes for each table and renaming the tpep/ltep in the yellow/green tables to match with the fhv/fhvhv ones.
###When we combine the tables, we want to know the source table for each piece of data. the {lit} makes each column have the source table name so you know where the data comes from

In [0]:
yellow_df = spark.read.parquet(yellow_input_path) \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('yellow'))

green_df = spark.read.parquet(green_input_path) \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('green')) 

fhv_df = spark.read.parquet(fhv_input_path) \
    .withColumn('taxi_type', F.lit('fhv')) \
    .withColumn('fare_amount', F.lit(None).cast('double'))
    
fhvhv_df = spark.read.parquet(fhvhv_input_path) \
    .withColumn('taxi_type', F.lit('fhvhv')) \
    .withColumnRenamed('base_passenger_fare', 'fare_amount') 
    

###{columns}  function returns a list of the column names. {Set} converts the lists into sets so you can use the '&'. The "&" returns the common columns between specified sets. 

In [0]:
common_columns = set(yellow_df.columns) & set(green_df.columns) & set(fhv_df.columns) & set(fhvhv_df.columns)
print(common_columns)

###Reassigning the dataframes to have the common columns

In [0]:
common_cols = ['DOLocationID', 'pu_day_of_week', 'do_day_of_week', 'dropoff_datetime', 
               'pu_month', 'do_month', 'pickup_datetime', 'is_do_unknown', 
               'pu_hour_of_day', 'do_hour_of_day', 'trip_duration', 
               'pu_year', 'do_year', 'is_pu_unknown', 'PULocationID', 'taxi_type', 'fare_amount']

yellow_df = yellow_df.select(common_cols)
green_df = green_df.select(common_cols)
fhv_df = fhv_df.select(common_cols)
fhvhv_df = fhvhv_df.select(common_cols)

###once the tables have the common columns(above cell), you can now union them. Union only works if the tables have the same columns

In [0]:
all_trips_combined_df = yellow_df\
    .union(green_df)\
    .union(fhv_df)\
    .union(fhvhv_df)

In [0]:
pu_zone = zone_df \
    .withColumnRenamed('Zone', 'pickup_zone') \
    .withColumnRenamed('Borough', 'pickup_borough') \
    .withColumnRenamed('LocationID', 'pu_id') \
    .withColumnRenamed('service_zone', 'pickup_service_zone')

In [0]:
do_zone = zone_df \
    .withColumnRenamed('Zone', 'dropoff_zone') \
    .withColumnRenamed('Borough', 'dropoff_borough') \
    .withColumnRenamed('LocationID', 'do_id')\
    .withColumnRenamed('service_zone', 'dropoff_service_zone')

In [0]:
all_trips_df = all_trips_combined_df\
    .join(pu_zone, all_trips_combined_df.PULocationID == pu_zone.pu_id, 'left')\
    .join(do_zone, all_trips_combined_df.DOLocationID == do_zone.do_id, 'left')\
    .drop("pu_id") \
    .drop("do_id")

all_trips_df.printSchema()

###Saving this as a table in the published layer as 'all_trips'

In [0]:
all_trips_output_path = 's3://ims-dbr-published/all_trips/'

In [0]:
all_trips_df = all_trips_df.select('PULocationID', 'pu_hour_of_day', 'pu_day_of_week', 'pu_month', 'pu_year', 'is_pu_unknown', 'pickup_datetime', 'pickup_borough', 'pickup_zone', 'pickup_service_zone', 
'DOLocationID', 'do_hour_of_day', 'do_day_of_week', 'do_month', 'do_year', 'is_do_unknown', 'dropoff_datetime', 'dropoff_borough', 'dropoff_zone', 'dropoff_service_zone', 'trip_duration', 'taxi_type', 'fare_amount')

In [0]:
all_trips_df.write \
  .mode("overwrite") \
  .parquet(all_trips_output_path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nytaxi.published.all_trips
USING PARQUET
LOCATION 's3://ims-dbr-published/all_trips/';

In [0]:
%sql
select * from nytaxi.published.all_trips limit 10